# 07a — pLLM embedding feature extraction (ESM hidden states) · local

**Purpose.** Extract per-variant **embedding-delta** features from the ESM-family ladder for all
4,783 single missense TEM-1 variants — the GPU half of the *supervised* pLLM arm (arm-3). The local
notebooks `07_EDA_pllm_embeddings` and `08_pllm_supervised_benchmark` read the parquet this notebook
produces (`data/features/plm_embeddings/pllm_embeddings.parquet`, via `paths.py`). It is the
embedding counterpart to the zero-shot score extractor `05a`.

**Feature blocks per model** (D035/D036): `delta_site` (emb(mut)[site] − emb(wt)[site], the primary
variant-specific feature), `delta_pooled` (mean over residues), `delta_local` (±7-residue window).

> **⚠️ >20-minute rule → run this on Colab.** Embedding extraction is GPU-heavy (one forward pass
> per variant × 7 models; 3B needs an A100). Per project convention (`CONVENTIONS.md`) the GPU step
> runs as the self-contained twin **`1.5 - Colab notebooks/07a_pllm_embedding_extraction_colab.ipynb`**.
> This local copy is the runnable reference: on a machine with a CUDA GPU it runs end-to-end and
> drops the parquet straight into `paths.py`'s `FEATURES_PLM_EMB`; on a CPU-only machine the driver
> cell **stops early with a message** rather than grinding for hours. Either way, once
> `pllm_embeddings.parquet` is in place, `07` and `08` pick it up.

**Model ladder** (D010–D013): ESM-1b, ESM-1v, ESM-2 {150M, 650M, 3B} (HF `EsmModel` hidden states),
ESM-C {300M, 600M} (EvolutionaryScale SDK). WT is embedded **once** per model and reused for all
4,783 deltas.


In [ ]:
# 1. self-contained: resolve project root via .projectroot, wire paths.py, check GPU
import sys, os, json, gzip, base64, re, time, gc, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root)); from paths import *

import numpy as np, pandas as pd, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = FEATURES_PLM_EMB; OUT.mkdir(parents=True, exist_ok=True)   # per-model checkpoints + final live here
CKPT = OUT/"_checkpoints"; CKPT.mkdir(exist_ok=True)
print("root:", root, "| device:", DEVICE)
if DEVICE == "cpu":
    print("\n*** NO CUDA GPU — this notebook is GPU-heavy (see the >20-min rule note above).")
    print("*** Run 1.5 - Colab notebooks/07a_pllm_embedding_extraction_colab.ipynb on an L4/A100,")
    print("*** then drop out/pllm_embeddings.parquet at:", (OUT/'pllm_embeddings.parquet'))


In [ ]:
# 2. dependencies (only if missing) — transformers/torch for ESM-1/2 hidden states via EsmModel;
# esm (EvolutionaryScale) for ESM-C. installing NOT fair-esm avoids the `import esm` namespace clash.
# on the project's betalactam env these are already present; uncomment to install on a fresh GPU box.
# %pip -q install "transformers>=4.40" "torch" "esm>=3.1.1"
try:
    import transformers  # noqa
    print("transformers", transformers.__version__)
except Exception:
    print("transformers not importable — install before running on a GPU box (see commented line)")


In [ ]:
# 3. config + variant slice — read from the on-disk modeling dataset (local: no embedded blob)
WT = "MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRVDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEPELNEAIPNDERDTTMPAAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWFIADKSGAGERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW"
df_lbl = pd.read_parquet(PROCESSED/"traditional_ml_aa_identity"/"modeling_dataset.parquet")
variants = df_lbl[["seq_id","position","wt_aa","mut_aa"]].sort_values(
    ["position","mut_aa"]).reset_index(drop=True)
L = len(WT)

# boundary checks (CLAUDE.md): position-1 indexes WT; WT residue there must equal wt_aa
seqidx = variants.position.values - 1
assert (seqidx>=0).all() and (seqidx<L).all(), "position out of range for WT"
assert all(WT[i]==aa for i,aa in zip(seqidx, variants.wt_aa.values)), "wt_aa != WT[pos-1]"
assert variants.seq_id.is_unique, "duplicate seq_id"
assert df_lbl.wt_seq.iloc[0]==WT, "WT constant != dataset wt_seq"

def _mut_seq(pos, ma): return WT[:pos-1] + ma + WT[pos:]
MUT_SEQS = [ _mut_seq(int(p), m) for p,m in zip(variants.position.values, variants.mut_aa.values) ]
assert all(len(s)==L for s in MUT_SEQS)
assert all(s[int(p)-1]==m for s,p,m in zip(MUT_SEQS, variants.position.values, variants.mut_aa.values))

TOK_IDX = variants.position.values.astype(int)   # BOS at 0 => residue i(0-based) at token i+1 => 1-based pos == token idx
WINDOW = 7
print(f"variants={len(variants)} positions={variants.position.nunique()} L={L}")


In [ ]:
# 4. HF ESM-1/2 embedder (last-layer hidden states)
# ESM HF tokenizers prepend CLS(BOS) and append EOS, so sequence index i maps to token index i+1.
# Every sequence here is exactly length L, so a batch needs NO padding: real residues are always
# tokens [1 .. L], BOS at 0, EOS at L+1. Embed WT once; delta each mutant against it.
from transformers import AutoTokenizer, EsmModel

@torch.no_grad()
def embed_hf(model_id, dtype=torch.float16, batch=8):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = EsmModel.from_pretrained(model_id, torch_dtype=dtype).to(DEVICE).eval()

    def _hidden(seq_list):
        enc = tok(seq_list, return_tensors="pt").to(DEVICE)   # all len L -> [B, L+2], no padding
        assert enc["input_ids"].shape[1] == L+2, "unexpected token length (padding?)"
        return model(**enc).last_hidden_state.float()         # [B, L+2, D]

    # sanity: token at index i+1 decodes to WT[i]
    enc0 = tok(WT, return_tensors="pt").to(DEVICE)
    assert tok.convert_ids_to_tokens(int(enc0["input_ids"][0,1]))==WT[0], "token offset != +1"

    # --- WT once ---
    wt_full = _hidden([WT])[0].cpu().numpy()          # [L+2, D]
    D = wt_full.shape[-1]
    wt_pooled = wt_full[1:L+1].mean(0)                # real residues only

    site   = np.empty((len(variants), D), dtype=np.float32)
    pooled = np.empty((len(variants), D), dtype=np.float32)
    local  = np.empty((len(variants), D), dtype=np.float32)

    for s in range(0, len(variants), batch):
        h = _hidden(MUT_SEQS[s:s+batch]).cpu().numpy()        # [b, L+2, D]
        for j in range(h.shape[0]):
            k = s+j; ti = int(TOK_IDX[k])
            site[k]   = h[j, ti] - wt_full[ti]
            pooled[k] = h[j, 1:L+1].mean(0) - wt_pooled
            lo = max(1, ti-WINDOW); hi = min(L, ti+WINDOW)    # token span within real residues [1, L]
            local[k]  = h[j, lo:hi+1].mean(0) - wt_full[lo:hi+1].mean(0)
        if s % (batch*40) == 0:
            print(f"  {model_id.split('/')[-1]}: {min(s+batch,len(variants))}/{len(variants)}", flush=True)

    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return dict(delta_site=site, delta_pooled=pooled, delta_local=local)

print("HF embedder ready")


In [ ]:
# 5. ESM-C embedder (EvolutionaryScale SDK) — hidden states via LogitsConfig(return_embeddings=True)
# ESM-C is BOS/EOS-bracketed like HF, so seq index i -> token index i+1. We feed token tensors
# (encode the string), not "_"-masked strings — embeddings need the real residue present.
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, ESMProteinTensor, LogitsConfig

@torch.no_grad()
def embed_esmc(model_key):
    client = ESMC.from_pretrained(model_key).to(DEVICE).eval()

    def _emb(seq):
        t = client.encode(ESMProtein(sequence=seq)).sequence.to(DEVICE)   # token ids [L+2]
        assert t.ndim==1 and t.shape[0]==L+2, f"ESM-C encoded len {tuple(t.shape)} != {L+2}"
        out = client.logits(ESMProteinTensor(sequence=t), LogitsConfig(sequence=True, return_embeddings=True))
        h = out.embeddings                                     # [1, L+2, D]
        assert h is not None, "ESM-C returned no embeddings — check return_embeddings support"
        return h[0].float().cpu().numpy()                      # [L+2, D]

    wt_full = _emb(WT); D = wt_full.shape[-1]
    wt_pooled = wt_full[1:L+1].mean(0)                         # real residues only (drop BOS/EOS)

    site  = np.empty((len(variants), D), dtype=np.float32)
    pooled= np.empty((len(variants), D), dtype=np.float32)
    local = np.empty((len(variants), D), dtype=np.float32)
    for k in range(len(variants)):
        h = _emb(MUT_SEQS[k])                                  # [L+2, D]
        ti = int(TOK_IDX[k])
        site[k]   = h[ti] - wt_full[ti]
        pooled[k] = h[1:L+1].mean(0) - wt_pooled
        lo = max(1, ti-WINDOW); hi = min(L, ti+WINDOW)
        local[k]  = h[lo:hi+1].mean(0) - wt_full[lo:hi+1].mean(0)
        if k % 800 == 0:
            print(f"  {model_key}: {k}/{len(variants)}", flush=True)

    del client; gc.collect(); torch.cuda.empty_cache()
    return dict(delta_site=site, delta_pooled=pooled, delta_local=local)

print("ESM-C embedder ready")


In [ ]:
# 6. assemble a per-model wide frame: one column per {model}__{block}__d{0000..} keyed by seq_id
def blocks_to_frame(key, blocks):
    cols = {"seq_id": variants.seq_id.values}
    for block, arr in blocks.items():                # delta_site / delta_pooled / delta_local
        D = arr.shape[1]
        for d in range(D):
            cols[f"{key}__{block}__d{d:04d}"] = arr[:, d]
    df = pd.DataFrame(cols)
    # boundary checks: finite, right shape, seq_id aligned
    assert df.seq_id.tolist() == variants.seq_id.tolist(), "seq_id order drifted"
    vals = df.drop(columns=["seq_id"]).values
    assert np.isfinite(vals).all(), f"non-finite embeddings in {key}"
    return df
print("assembler ready")


In [ ]:
# 7. run the ladder — checkpoint each model to _checkpoints/ so a crash never loses prior models.
# on CPU this stops early per the >20-min rule; use the Colab twin instead.
if DEVICE == "cpu":
    raise SystemExit("CPU device — skipping extraction. Run the Colab twin and drop the parquet in place.")

LADDER = [
    ("esm1b",     "hf",   "facebook/esm1b_t33_650M_UR50S"),
    ("esm1v",     "hf",   "facebook/esm1v_t33_650M_UR90S_1"),
    ("esm2_150m", "hf",   "facebook/esm2_t30_150M_UR50D"),
    ("esm2_650m", "hf",   "facebook/esm2_t33_650M_UR50D"),
    ("esm2_3b",   "hf",   "facebook/esm2_t36_3B_UR50D"),    # A100 recommended
    ("esmc_300m", "esmc", "esmc_300m"),
    ("esmc_600m", "esmc", "esmc_600m"),
]
BATCH = {"esm2_3b": 2, "esm2_650m": 6, "esm1b": 6, "esm1v": 6, "esm2_150m": 16}

status = {}
for key, kind, ref in LADDER:
    ck = CKPT/f"emb_{key}.parquet"
    if ck.exists():
        print(f"[skip] {key} — checkpoint exists"); status[key]="cached"; continue
    print(f"[run ] {key} ({ref}) ...", flush=True); t0=time.time()
    try:
        blocks = embed_hf(ref, batch=BATCH.get(key,8)) if kind=="hf" else embed_esmc(ref)
        frame = blocks_to_frame(key, blocks)
        frame.to_parquet(ck, index=False)
        status[key]=f"ok {time.time()-t0:.0f}s"
        print(f"[done] {key} in {time.time()-t0:.0f}s  cols={frame.shape[1]-1} "
              f"(D={blocks['delta_site'].shape[1]} x 3 blocks)")
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        status[key]="OOM — needs a bigger GPU (A100 for 3B); skipped"
        print(f"[OOM ] {key}: out of memory — use A100 for this model, or skip it")
    except Exception as e:
        status[key]=f"ERROR: {e}"; print(f"[fail] {key}: {e}")
print("\nstatus:", json.dumps(status, indent=0))


In [ ]:
# 8. merge per-model checkpoints -> the final embeddings parquet the local pipeline reads
parts = sorted(CKPT.glob("emb_*.parquet"))
assert parts, "no per-model checkpoints found — run cell 7 on a GPU, or use the Colab twin"
merged = variants[["seq_id"]].copy()
for p in parts:
    merged = merged.merge(pd.read_parquet(p), on="seq_id", how="left")
feat_cols = [c for c in merged.columns if c!="seq_id"]
assert len(merged)==len(variants) and merged.seq_id.is_unique
bad = [c for c in feat_cols if merged[c].isna().all()]
assert not bad, f"all-NaN embedding columns: {bad[:5]} ..."
assert np.isfinite(merged[feat_cols].values).all(), "non-finite values in merged embeddings"
merged.attrs = {}
FINAL = FEATURES_PLM_EMB/"pllm_embeddings.parquet"
merged.to_parquet(FINAL, index=False)
import collections
cc = collections.Counter(c.rsplit("__",1)[0] for c in feat_cols)
print("wrote", FINAL.relative_to(BASE_DIR), merged.shape)
for k in sorted(cc): print(f"  {k}: {cc[k]} dims")


In [ ]:
# 9. quick self-check — a real embedding delta should be LARGER at the mutated site than at a
# random distant control residue (the substitution perturbs its own position most). also the
# delta_site block should carry more per-variant variance than the whole-sequence pooled delta,
# which barely moves for a single substitution (the reason delta_site is the primary feature, D035).
rng = np.random.default_rng(42)
for key in sorted({c.split("__")[0] for c in feat_cols}):
    sc = [c for c in feat_cols if c.startswith(f"{key}__delta_site__")]
    pc = [c for c in feat_cols if c.startswith(f"{key}__delta_pooled__")]
    if not sc: continue
    site_norm   = np.linalg.norm(merged[sc].values, axis=1)
    pooled_norm = np.linalg.norm(merged[pc].values, axis=1)
    print(f"{key:10s} |delta_site| median={np.median(site_norm):.3f}  "
          f"|delta_pooled| median={np.median(pooled_norm):.3f}  "
          f"ratio={np.median(site_norm)/max(np.median(pooled_norm),1e-9):.1f}x")


## 9. Done

`data/features/plm_embeddings/pllm_embeddings.parquet` now holds every model's three delta blocks,
one column per `{model}__{block}__d{0000..}`, keyed by `seq_id`. Run `07_EDA_pllm_embeddings` next,
then `08_pllm_supervised_benchmark`.

**Guardrail carried into 08 (D037).** Wide matrix (3 blocks × up to 7 models). `08` reduces with PCA
fit **inside each CV fold** (train only) and asserts no `seq_id`/`position`/label-derived column
enters the feature matrix.
